# Research data supporting "Practical and improved density functionals for computational catalysis on metal surfaces"

This notebook accompanies our paper: **Practical and improved density functionals for computational catalysis on metal surfaces**. It can be found on GitHub at https://github.com/benshi97/Data_NSCDFT_for_Metals and explored interactively on [Colab](https://colab.research.google.com/github/benshi97/Data_NSCDFT_for_Metals/blob/main/analyse.ipynb).

### Abstract

Density functional theory (DFT) has been critical towards our current atomistic understanding of catalysis on transition-metal surfaces.
It has opened new paradigms in rational catalyst design, predicting fundamental properties, like the adsorption energy and reaction barriers, linked to catalytic performance.
However, such applications depend sensitively on the predictive accuracy of DFT and achieving experimental-level reliability, so-called transition-metal chemical accuracy (13 kJ/mol), remains challenging for present density functional approximations (DFAs) or even methods beyond DFT.
We introduce a new framework for designing DFAs tailored towards studying molecules adsorbed on transition-metal surfaces, building upon recent developments in non-self-consistent DFAs.
We propose two functionals within this framework, demonstrating that transition-metal chemical accuracy can be achieved across a diverse set of 39 adsorption reactions while delivering consistent performance for 17 barrier heights.
Moreover, we show that these non-self-consistent DFAs address qualitative failures that challenge current self-consistent DFAs, such as CO adsorption on Pt(111) and graphene on Ni(111).
The resulting functionals are computationally practical and compatible with existing DFT codes, with scripts and workflows provided to use them.
Looking ahead, this framework establishes a path toward developing more accurate and sophisticated DFAs for computational heterogeneous catalysis and beyond.


## Table of Contents

1. [Lattice energies of X23 dataset](#lattice-energies-of-x23-dataset)
2. [Analysis DMC costs](#analysis-dmc-costs)
3. [Lattice and relative energies of ICE13](#lattice-and-relative-energies-of-ice13)
4. [Optimizing B86bPBE50-revXDM](#optimizing-b86bpbe50-revxdm)
5. [Relative energies of pharmaceutical polymorphs](#relative-energies-of-pharmaceutical-polymorphs)
6. [DFT Benchmarks](#dft-benchmarks)

In [5]:
# Check if we are in Google Colab environment
try:
    import google.colab
    IN_COLAB = True
    usetex = False
    import os
    import subprocess
    import sys
except:
    import os
    IN_COLAB = False
    if os.path.expanduser('~') == '/Users/bshi':
        usetex = True
    else:
        usetex = False

if usetex:
    def textbf(text):
        return r'\textbf{' + text + '}'
else:
    def textbf(text):
        return text

# If in Google Colab, install the necessary data and set up the necessary environment
if IN_COLAB == True:
    repo_url = "https://github.com/benshi97/Data_NSCDFT_for_Metals"

    # Clone the repository
    %cd /content
    !rm -rf /content/Data_LNOMBECC
    !git clone {repo_url}
    %cd /content/Data_LNOMBECC

replot_analysis = False

# Import necessary functions
from Scripts.analysis_scripts import *
from Scripts.plot_scripts import *

if usetex == True:
    textrue_import()
else:
    texfalse_import()

dft_xc_list = [
    "01_PBE",
    "02_PBEsol",
    "03_RPBE",
    "04_revPBE",
    "05_BLYP",
    "06_r2SCAN",
    "07_MS2",
    "08_revTPSS",
    "09_PBED3",
    "10_PBEDDsC",
    "11_RPBED3",
    "12_optPBEvdW",
    "13_revvdWDF2",
    "14_r2SCANrVV10"
]

method_dict = {
    '01_PBE': {'label': 'PBE', 'color': 'orange'},
    '02_PBEsol': {'label': 'PBEsol', 'color': 'purple'},
    '03_RPBE': {'label': 'RPBE', 'color': 'cyan'},
    '04_revPBE': {'label': 'revPBE', 'color': 'magenta'},
    '05_BLYP': {'label': 'BLYP', 'color': 'lime'},
    '06_r2SCAN': {'label': r'r$^2$SCAN', 'color': 'pink'},
    '07_MS2': {'label': 'MS2', 'color': 'teal'},
    '08_revTPSS': {'label': 'revTPSS', 'color': 'lavendar'},
    '09_PBED3': {'label': 'PBE-D3', 'color': 'brown'},
    '10_PBEDDsC': {'label': 'PBE-DdsC', 'color': 'beige'},
    '11_RPBED3': {'label': 'RPBE-D3', 'color': 'maroon'},
    '12_optPBEvdW': {'label': 'optPBE-vdW', 'color': 'apricot'},
    '13_revvdWDF2': {'label': 'revvdW-DF2', 'color': 'navy'},
    '14_r2SCANrVV10': {'label': r'r$^2$SCAN+rVV10', 'color': 'olive'},
    'RPA': {'label': 'RPA@PBE', 'color': 'blue'},
    'BEEFvdW': {'label': 'BEEF-vdW', 'color': 'red'},
    'hBEEFvdW': {'label': 'hBEEF-vdW', 'color': 'green'},
    'dhBEEFvdW': {'label': 'dhBEEF-vdW', 'color': 'yellow'},
}


# CE39 dataset

## Experimental data

In [6]:
raw_data = { # Taken from Surf. Sci. 640, 36–44 (2015).
    "#": list(range(1, 40)),
    "Surface reaction": [
        "CO + Ni(111) → CO/Ni(111)",
        "CO + Pt(111) → CO/Pt(111)",
        "CO + Pd(111) → CO/Pd(111)",
        "CO + Pd(100) → CO/Pd(100)",
        "CO + Rh(111) → CO/Rh(111)",
        "CO + Ir(111) → CO/Ir(111)",
        "CO + Cu(111) → CO/Cu(111)",
        "CO + Ru(001) → CO/Ru(001)",
        "CO + Co(001) → CO/Co(001)",
        "NO + Ni(100) → N/Ni(100) + O/Ni(100)",
        "NO + Pt(111) → NO/Pt(111)",
        "NO + Pd(111) → NO/Pd(111)",
        "NO + Pd(100) → NO/Pd(100)",
        "O2 + Ni(111) → 2O/Ni(111)",
        "O2 + Ni(100) → 2O/Ni(100)",
        "O2 + Pt(111) → 2O/Pt(111)",
        "O2 + Rh(100) → 2O/Rh(100)",
        "H2 + Pt(111) → 2H/Pt(111)",
        "H2 + Ni(111) → 2H/Ni(111)",
        "H2 + Ni(100) → 2H/Ni(100)",
        "H2 + Rh(111) → 2H/Rh(111)",
        "H2 + Pd(111) → 2H/Pd(111)",
        "I2 + Pt(111) → 2I/Pt(111)",
        "CH2I2 + Pt(111) → CH/Pt(111) + H/Pt(111) + 2I/Pt(111)",
        "CH3I + Pt(111) → CH3/Pt(111) + I/Pt(111)",
        "NH3 + Cu(100) → NH3/Cu(100)",
        "CH3I + Pt(111) → CH3I/Pt(111)",
        "CH3OH + Pt(111) → CH3OH/Pt(111)",
        "CH4 + Pt(111) → CH4/Pt(111)",
        "C2H6 + Pt(111) → C2H6/Pt(111)",
        "C3H8 + Pt(111) → C3H8/Pt(111)",
        "C4H10 + Pt(111) → C4H10/Pt(111)",
        "C6H6 + Pt(111) → C6H6/Pt(111)",
        "C6H6 + Cu(111) → C6H6/Cu(111)",
        "C6H6 + Ag(111) → C6H6/Ag(111)",
        "C6H6 + Au(111) → C6H6/Au(111)",
        "C6H10 + Pt(111) → C6H10/Pt(111)",
        "H2O + Pt(111) → H2O/Pt(111)",
        "H2O + OPt(111) → H2OOH/Pt(111)"
    ],
    "Coverage (ML)": [
        "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/8", "1/4", "1/4", "1/4", "1/8",
        "1/8", "1/18", "1/8", "1/8", "1/8", "1/8", "1/8", "1/8", "1/4", "1/4", "1/4", "1/4", "1/4",
        "1/4", "1/4", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/4", "1/2"
    ],
    "Reaction enthalpy (kJ/mol)": [
        -122, -120, -143, -155, -139, -158, -53, -158, -115, -290, -114, -179, -161, -480,
        -530, -208, -358, -72, -94, -94, -70, -88, -312, -473, -212, -57, -84.5, -56, -15,
        -28.5, -41.3, -50.8, -164, -68, -63, -72, -122, -51.3, -57.4
    ],
    "Temp. (K)": [
        350, 340, 450, 430, 500, 420, 350, 475, 370, 300, 300, 520, 300, 300,
        300, 515, 300, 300, 370, 370, 325, 370, 0, 210, 320, 235, 100, 100,
        63, 106, 139, 171, 300, 225, 210, 230, 100, 120, 150
    ],
    "Reaction energy (kJ/mol)": [
        -119, -117, -139, -151, -135, -154, -52, -154, -112, -288, -112, -175, -159, -479,
        -528, -204, -356, -70, -91, -91, -67, -85, -230, -471, -209, -55, -83.7, -55,
        -14.5, -27.6, -40.1, -49.4, -300, -66, -61, -70, -121, -50.3, -56.2
    ],
    "Reaction energy without ZPE (kJ/mol)": [
        -124, -124, -144, -157, -142, -164, -57, -161, -119, -299,
        -119, -182, -163, -485, -530, -208, -355, -72, -100, -87,
        -72, -90, -313, -455, -209
    ] + [
        -60, -84, -55, -14, -27, -39, -48, -162, -66, -61,
        -70, -123, -55, -66
    ],
    "Weight": [ 1, 1, 1, 1, 1, 1, 1, 1, 1,0.5, 1, 1, 1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5,0.5, 0.5, 0.5, 0.5, 0.25, 0.5] + [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,1]
}

ce39_dataset_dict = {}

calc_molecule_list = []
calc_molecule_surface_list = []

for ce39_idx in range(39):


    if 'Co' in raw_data['Surface reaction'][ce39_idx] or 'Ru' in raw_data['Surface reaction'][ce39_idx]:
        crystal_type = 'hcp'
    else:
        crystal_type = 'fcc'

    # Set the supercell_size based on coverage
    if raw_data['Coverage (ML)'][ce39_idx] in ['1/4','1/8']:
        surface_sc_size = '2x2'
    else:
        surface_sc_size = '3x3'

    surface_type = raw_data['Surface reaction'][ce39_idx].split()[2].replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')
    system_molecule = raw_data['Surface reaction'][ce39_idx].split()[0]
    molecule_surface_system_list = []
    for molecule_surface_system in raw_data['Surface reaction'][ce39_idx].split('→')[1].split('+'):
        if molecule_surface_system.strip()[0].isdigit():
            molecule_surface_system_list += [molecule_surface_system.strip()[1:].replace('/','-').replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')]*int(molecule_surface_system.strip()[0])
        else:
            molecule_surface_system_list += [molecule_surface_system.strip().replace('/','-').replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')]


    ce39_dataset_dict[f'{ce39_idx+1:02d}'] = {'Reaction': raw_data['Surface reaction'][ce39_idx], 'MAD Weight': raw_data['Weight'][ce39_idx], 'Reactant Molecule': [system_molecule], 'Reactant Surface': [surface_type], 'Products': molecule_surface_system_list, 'Reaction Enthalpy': int(raw_data["Reaction enthalpy (kJ/mol)"][ce39_idx]), 'Reaction Energy': int(raw_data["Reaction energy (kJ/mol)"][ce39_idx]),'Reaction Energy without ZPE': int(raw_data["Reaction energy without ZPE (kJ/mol)"][ce39_idx])}

    for molecule_surface_system in molecule_surface_system_list:
        if molecule_surface_system not in calc_molecule_surface_list:
            calc_molecule_surface_list += [molecule_surface_system]
    
    if system_molecule not in calc_molecule_list:
        calc_molecule_list += [system_molecule]

ce39_dataset_df = pd.DataFrame.from_dict(ce39_dataset_dict, orient='index')
display(ce39_dataset_df)

,Reaction,MAD Weight,Reactant Molecule,Reactant Surface,Products,Reaction Enthalpy,Reaction Energy,Reaction Energy without ZPE
01,CO + Ni(111) → CO/Ni(111),1.00,[CO],[Ni_fcc_111_2x2],[CO-Ni_fcc_111_2x2],-122,-119,-124
02,CO + Pt(111) → CO/Pt(111),1.00,[CO],[Pt_fcc_111_2x2],[CO-Pt_fcc_111_2x2],-120,-117,-124
03,CO + Pd(111) → CO/Pd(111),1.00,[CO],[Pd_fcc_111_2x2],[CO-Pd_fcc_111_2x2],-143,-139,-144
04,CO + Pd(100) → CO/Pd(100),1.00,[CO],[Pd_fcc_100_2x2],[CO-Pd_fcc_100_2x2],-155,-151,-157
05,CO + Rh(111) → CO/Rh(111),1.00,[CO],[Rh_fcc_111_2x2],[CO-Rh_fcc_111_2x2],-139,-135,-142
06,CO + Ir(111) → CO/Ir(111),1.00,[CO],[Ir_fcc_111_2x2],[CO-Ir_fcc_111_2x2],-158,-154,-164
07,CO + Cu(111) → CO/Cu(111),1.00,[CO],[Cu_fcc_111_2x2],[CO-Cu_fcc_111_2x2],-53,-52,-57
08,CO + Ru(001) → CO/Ru(001),1.00,[CO],[Ru_hcp_001_2x2],[CO-Ru_hcp_001_2x2],-158,-154,-161
09,CO + Co(001) → CO/Co(001),1.00,[CO],[Co_hcp_001_2x2],[CO-Co_hcp_001_2x2],-115,-112,-119
10,NO + Ni(100) → N/Ni(100) + O/Ni(100),0.50,[NO],[Ni_fcc_100_2x2],"[N-Ni_fcc_100_2x2, O-Ni_fcc_100_2x2]",-290,-288,-299


## DFT estimates

### BEEF-vdW

In [12]:
# Calculate the adsorption energy from BEEF-vdW with a 4L and 6L slab
vasp_method = 'vasp'

ce39_eads_dict = {ce39_idx: {'Reaction': '', 'Experiment': 0, 'BEEFvdW (4L)': 0, 'BEEFvdW (6L)': 0} for ce39_idx in ce39_dataset_dict}

for method in ['4L','6L']:
    if method == '4L':
        root_folder = f'Data/CE39/Eads_BEEFvdW_4L'
    elif method == '6L':
        root_folder = f'Data/CE39/Eads_BEEFvdW_6L'

    for ce39_idx in ce39_dataset_dict:
        reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
        reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
        products = ce39_dataset_dict[ce39_idx]['Products']
        sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
        num_products = len(products)

        total_product_energy = 0
        for product_idx in range(num_products):
            total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/{method}_{products[product_idx]}/OUTCAR.gz', code='vasp')

        if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR.gz', code='vasp')
            reactant_surface_energy = get_energy(f'{root_folder}/Surface/{method}_{reactant_surface}/OUTCAR.gz', code='vasp')
            ce39_eads_dict[ce39_idx][f'BEEFvdW ({method})'] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
        else:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR.gz', code='vasp')
            reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/{method}_O-Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp')  - (1/9)*get_energy(f'{root_folder}/Surface/{method}_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') 
            ce39_eads_dict[ce39_idx][f'BEEFvdW ({method})'] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV

        ce39_eads_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']
        ce39_eads_dict[ce39_idx]['Reaction'] = ce39_dataset_dict[ce39_idx]['Reaction']

# Convert to DataFrame
ce39_eads_df = pd.DataFrame.from_dict(ce39_eads_dict, orient='index')
mad_dict = {method: np.mean(abs(ce39_eads_df[method] - ce39_eads_df['Experiment']) * ce39_dataset_df['MAD Weight']) for method in ['BEEFvdW (4L)', 'BEEFvdW (6L)']}
# Add an additional row at the bottom that is the MAD for each method
ce39_eads_df.loc['MAD'] = ['MAD', 0, mad_dict['BEEFvdW (4L)'], mad_dict['BEEFvdW (6L)']]

# Round to nearest integer and convert to integer
ce39_eads_df = ce39_eads_df.round(0).astype({'Experiment': 'Int64', 'BEEFvdW (4L)': 'Int64', 'BEEFvdW (6L)': 'Int64', 'Reaction': str}) 

display(ce39_eads_df)

,Reaction,Experiment,BEEFvdW (4L),BEEFvdW (6L)
01,CO + Ni(111) → CO/Ni(111),-124,-150,-151
02,CO + Pt(111) → CO/Pt(111),-124,-133,-135
03,CO + Pd(111) → CO/Pd(111),-144,-158,-148
04,CO + Pd(100) → CO/Pd(100),-157,-152,-150
05,CO + Rh(111) → CO/Rh(111),-142,-161,-163
06,CO + Ir(111) → CO/Ir(111),-164,-172,-179
07,CO + Cu(111) → CO/Cu(111),-57,-52,-50
08,CO + Ru(001) → CO/Ru(001),-161,-162,-156
09,CO + Co(001) → CO/Co(001),-119,-135,-137
10,NO + Ni(100) → N/Ni(100) + O/Ni(100),-299,-393,-385


## Comparison between NSC-DFAs, RPA and DFT

In [23]:
ce39_int_dict

{'01': {'Reaction': '',
  'Experiment': 0,
  'RPA': 0,
  'BEEFvdW': -43432.31148274403,
  'hBEEFvdW': 0,
  'dhBEEFvdW': 0},
 '02': {'Reaction': '',
  'Experiment': 0,
  'RPA': 0,
  'BEEFvdW': -48142.3756564037,
  'hBEEFvdW': 0,
  'dhBEEFvdW': 0},
 '03': {'Reaction': '',
  'Experiment': 0,
  'RPA': 0,
  'BEEFvdW': -53792.24498080611,
  'hBEEFvdW': 0,
  'dhBEEFvdW': 0},
 '04': {'Reaction': '',
  'Experiment': 0,
  'RPA': 0,
  'BEEFvdW': -57692.22473322632,
  'hBEEFvdW': 0,
  'dhBEEFvdW': 0},
 '05': {'Reaction': '',
  'Experiment': 0,
  'RPA': 0,
  'BEEFvdW': -61477.401212591656,
  'hBEEFvdW': 0,
  'dhBEEFvdW': 0},
 '06': {'Reaction': '',
  'Experiment': 0,
  'RPA': 0,
  'BEEFvdW': -68724.95517271628,
  'hBEEFvdW': 0,
  'dhBEEFvdW': 0},
 '07': {'Reaction': '',
  'Experiment': 0,
  'RPA': 0,
  'BEEFvdW': -78192.09141928871,
  'hBEEFvdW': 0,
  'dhBEEFvdW': 0},
 '08': {'Reaction': '',
  'Experiment': 0,
  'RPA': 0,
  'BEEFvdW': -80151.46435180579,
  'hBEEFvdW': 0,
  'dhBEEFvdW': 0},
 '09': {

In [ ]:
# Analyse the interaction energy calculations
vasp_method = 'vasp'

ce39_int_dict = {ce39_idx: {'RPA': 0 , 'BEEFvdW': 0, 'hBEEFvdW': 0,'dhBEEFvdW': 0} for ce39_idx in ce39_dataset_dict}
final_ce39_eads_dict = {ce39_idx: {'Reaction': '', 'Experiment': 0, 'RPA': 0 , 'BEEFvdW': 0, 'hBEEFvdW': 0, 'dhBEEFvdW': 0} for ce39_idx in ce39_dataset_dict}

# Start by analyzing BEEFvdW
root_folder = "Data/CE39/Eint_BEEFvdW"
for ce39_idx in ce39_int_dict:
    reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
    reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
    products = ce39_dataset_dict[ce39_idx]['Products']
    sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
    num_products = len(products)

    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR.gz', code='vasp')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR.gz', code='vasp')
        reactant_surface_energy = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR.gz', code='vasp')
        ce39_int_dict[ce39_idx]['BEEFvdW'] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp')
        reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') 
        ce39_int_dict[ce39_idx]['BEEFvdW'] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV


# Analyze hBEEFvdW, dhBEEFvdW and RPA
root_folder = f'Data/CE39/Eint_hBEEFvdW'
rpa_root_folder = f'Data/CE39/Eint_RPA'

for ce39_idx in ce39_int_dict:
    reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
    reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
    products = ce39_dataset_dict[ce39_idx]['Products']
    sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
    num_products = len(products)
    calc_energy_dict = {
        '00': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '01': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '02': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '03_3': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0}
    }
    # hBEEF part
    for calc_type in calc_energy_dict:
        for product_idx in range(num_products):
            calc_energy_dict[calc_type]['total_product_energy'] += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_{calc_type}.gz', code='vasp')

        if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
            calc_energy_dict[calc_type]['reactant_molecule_energy'] = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['reactant_surface_energy'] = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['eint'] = (calc_energy_dict[calc_type]['total_product_energy'] - calc_energy_dict[calc_type]['reactant_molecule_energy'] - num_products * calc_energy_dict[calc_type]['reactant_surface_energy']) * 1000 / kjmol_to_meV
        else:
            calc_energy_dict[calc_type]['reactant_molecule_energy'] = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['reactant_surface_energy'] = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp') - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp') 
            calc_energy_dict[calc_type]['eint'] = (2 / 9 * calc_energy_dict[calc_type]['total_product_energy'] - calc_energy_dict[calc_type]['reactant_molecule_energy'] - calc_energy_dict[calc_type]['reactant_surface_energy']) * 1000 / kjmol_to_meV

    # RPA correlation part
    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_04_RPA.gz', code='vasp_rpa')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        reactant_surface_energy = get_energy(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        rpac_eint = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        reactant_surface_energy = (1/3)*get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa')- (1/9)*get_energy(f'{rpa_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa') 
        rpac_eint = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV

    # RPA EXX part
    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_02_EXX.gz', code='vasp')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_02_EXX.gz', code='vasp')
        reactant_surface_energy = get_energy(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_02_EXX.gz', code='vasp')
        rpax_eint = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp')
        reactant_surface_energy = (1/3)*get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp') - (1/9)*get_energy(f'{rpa_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp') 
        rpax_eint = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV


    beefx_eint = calc_energy_dict['02']['eint']
    exx_eint = calc_energy_dict['03_3']['eint']
    nlc_eint = calc_energy_dict['00']['eint'] - calc_energy_dict['01']['eint']
    beefc_eint = calc_energy_dict['01']['eint'] - calc_energy_dict['02']['eint']
    dh_x_frac = 0.25
    dh_c_frac = 0.15
    ce39_int_dict[ce39_idx]['dhBEEFvdW'] = dh_x_frac * exx_eint + (1-dh_x_frac)*beefx_eint + dh_c_frac*rpac_eint + (1-dh_c_frac)*beefc_eint + nlc_eint
    h_x_frac = 0.175
    ce39_int_dict[ce39_idx]['hBEEFvdW'] = h_x_frac * exx_eint + (1-h_x_frac)*beefx_eint + beefc_eint + nlc_eint
    ce39_int_dict[ce39_idx]['RPA'] = rpax_eint + rpac_eint

    final_ce39_eads_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']
    final_ce39_eads_dict[ce39_idx]['Reaction'] = ce39_dataset_dict[ce39_idx]['Reaction']


# Convert interaction energy to adsorption energy

for method in ['RPA','BEEFvdW','hBEEFvdW','dhBEEFvdW']:
    for ce39_idx in ce39_int_dict:
        final_ce39_eads_dict[ce39_idx][method] = ce39_int_dict[ce39_idx][method] - ce39_int_dict[ce39_idx]['BEEFvdW'] + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)']

# Calculate the adsorption energy using BEEF-vdW
vasp_method = 'vasp'

ce39_dft_xc_eads_dict = {ce39_idx: {'Reaction': '', 'Experiment': 0} | {dft_xc : 0 for dft_xc in dft_xc_list} for ce39_idx in ce39_dataset_dict}

for dft_xc in dft_xc_list:
    root_folder = f'Data/CE39/DFT_XC'
    for ce39_idx in ce39_dft_xc_eads_dict:
        try:
            reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
            reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
            products = ce39_dataset_dict[ce39_idx]['Products']
            sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
            num_products = len(products)

            total_product_energy = 0
            for product_idx in range(num_products):
                total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_{dft_xc}', code='vasp')

            if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
                reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR_{dft_xc}', code='vasp')
                reactant_surface_energy = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR_{dft_xc}', code='vasp')
                ce39_dft_xc_eads_dict[ce39_idx][dft_xc] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kJmol2meV + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)'] - ce39_eads_dict[ce39_idx]['BEEFvdW (4L)']
            else:
                reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR_{dft_xc}', code='vasp')
                reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_{dft_xc}', code='vasp')  - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_{dft_xc}', code='vasp') 
                ce39_dft_xc_eads_dict[ce39_idx][dft_xc] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kJmol2meV + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)'] - ce39_eads_dict[ce39_idx]['BEEFvdW (4L)']

            ce39_dft_xc_eads_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']
            ce39_dft_xc_eads_dict[ce39_idx]['Reaction'] = ce39_dataset_dict[ce39_idx]['Reaction']
        # print(ce39_int_dict[ce39_idx]['Reaction'], total_product_energy, reactant_molecule_energy, reactant_surface_energy, ce39_int_dict[ce39_idx]['RPA'])
        except:
            print(f"Error processing reaction {ce39_idx}: {ce39_dataset_dict[ce39_idx]['Reaction']}")
            continue


In [31]:
ce39_int_dict

{'01': {'RPA': -98.23393659927467,
  'BEEFvdW': -149.23116278004022,
  'hBEEFvdW': -137.52097467897727,
  'dhBEEFvdW': -138.3818920094924},
 '02': {'RPA': -133.06607768723273,
  'BEEFvdW': -133.40581215425186,
  'hBEEFvdW': -153.47950461277594,
  'dhBEEFvdW': -145.21359273906023},
 '03': {'RPA': -148.96420559795638,
  'BEEFvdW': -161.19553413357534,
  'hBEEFvdW': -176.61411829891932,
  'dhBEEFvdW': -162.16334667431383},
 '04': {'RPA': -148.68814393443623,
  'BEEFvdW': -154.54307498023314,
  'hBEEFvdW': -166.87065084836283,
  'dhBEEFvdW': -157.77709925984018},
 '05': {'RPA': -138.20635678900157,
  'BEEFvdW': -160.64965485484646,
  'hBEEFvdW': -176.2965190965569,
  'dhBEEFvdW': -167.65728552109357},
 '06': {'RPA': -150.42955637743157,
  'BEEFvdW': -170.76060664402814,
  'hBEEFvdW': -187.2633132040079,
  'dhBEEFvdW': -179.258674067891},
 '07': {'RPA': -44.39197146343825,
  'BEEFvdW': -50.58621219957913,
  'hBEEFvdW': -44.69272944522881,
  'dhBEEFvdW': -52.3064989743943},
 '08': {'RPA': -1

In [32]:
final_ce39_eads_dict

{'01': {'Reaction': 'CO + Ni(111) → CO/Ni(111)',
  'Experiment': -124,
  'RPA': -99.7097675673653,
  'BEEFvdW': 0,
  'hBEEFvdW': -138.9968056470679,
  'dhBEEFvdW': -139.85772297758302},
 '02': {'Reaction': 'CO + Pt(111) → CO/Pt(111)',
  'Experiment': -124,
  'RPA': -134.99852259862985,
  'BEEFvdW': 0,
  'hBEEFvdW': -155.41194952417305,
  'dhBEEFvdW': -147.14603765045734},
 '03': {'Reaction': 'CO + Pd(111) → CO/Pd(111)',
  'Experiment': -144,
  'RPA': -135.86954523412936,
  'BEEFvdW': 0,
  'hBEEFvdW': -163.5194579350923,
  'dhBEEFvdW': -149.0686863104868},
 '04': {'Reaction': 'CO + Pd(100) → CO/Pd(100)',
  'Experiment': -157,
  'RPA': -143.71941462565323,
  'BEEFvdW': 0,
  'hBEEFvdW': -161.90192153957983,
  'dhBEEFvdW': -152.8083699510572},
 '05': {'Reaction': 'CO + Rh(111) → CO/Rh(111)',
  'Experiment': -142,
  'RPA': -140.36643858141514,
  'BEEFvdW': 0,
  'hBEEFvdW': -178.45660088897046,
  'dhBEEFvdW': -169.81736731350713},
 '06': {'Reaction': 'CO + Ir(111) → CO/Ir(111)',
  'Experimen

In [ ]:
# Conver to DataFrame
ce39_int_df = pd.DataFrame.from_dict(ce39_int_dict, orient='index')
# Round all values to nearest integer and convert to int astype
ce39_int_df = ce39_int_df.round(0).astype({'Experiment': 'Int64', 'RPA': 'Int64', 'RPA_beef': 'Int64', 'rSE': 'Int64',  'RPA+rSE': 'Int64', 'RPA+EXX75': 'Int64','BEEFvdW': 'Int64','hBEEFvdW': 'Int64', 'Reaction': str}) 


metrics_methods = ['RPA', 'BEEFvdW', 'hBEEFvdW', 'dhBEEFvdW']

# Convert final_ce39_eads_dict to DataFrame
final_ce39_eads_df = pd.DataFrame.from_dict(final_ce39_eads_dict, orient='index')

# Calculate the adsorption energy for several DFT XC functionals
vasp_method = 'vasp'

ce39_dft_xc_eads_dict = {ce39_idx: {'Reaction': '', 'Experiment': 0} | {dft_xc : 0 for dft_xc in dft_xc_list} for ce39_idx in ce39_dataset_dict}

for dft_xc in dft_xc_list:
    root_folder = f'Data/CE39/DFT_XC'
    for ce39_idx in ce39_dft_xc_eads_dict:
        try:
            reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
            reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
            products = ce39_dataset_dict[ce39_idx]['Products']
            sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
            num_products = len(products)

            total_product_energy = 0
            for product_idx in range(num_products):
                total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_{dft_xc}.gz', code='vasp')

            if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
                reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR_{dft_xc}.gz', code='vasp')
                reactant_surface_energy = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR_{dft_xc}.gz', code='vasp')
                ce39_dft_xc_eads_dict[ce39_idx][dft_xc] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)'] - ce39_eads_dict[ce39_idx]['BEEFvdW (4L)']
            else:
                reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR_{dft_xc}.gz', code='vasp')
                reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_{dft_xc}.gz', code='vasp')  - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_{dft_xc}.gz', code='vasp') 
                ce39_dft_xc_eads_dict[ce39_idx][dft_xc] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)'] - ce39_eads_dict[ce39_idx]['BEEFvdW (4L)']

            ce39_dft_xc_eads_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']
            ce39_dft_xc_eads_dict[ce39_idx]['Reaction'] = ce39_dataset_dict[ce39_idx]['Reaction']
        except:
            print(f"Error processing reaction {ce39_idx}: {ce39_dataset_dict[ce39_idx]['Reaction']}")
            continue

metrics_methods = ['RPA', 'RPA_beef', 'RPA+rSE',  'RPA+EXX75', 'BEEFvdW', 'hBEEFvdW', 'dhBEEFvdW'] + dft_xc_list

# Add DFT XC results to final_ce39_eads_df
ce39_dft_xc_eads_df = pd.DataFrame.from_dict(ce39_dft_xc_eads_dict, orient='index')
# Round to nearest integer and convert to integer
ce39_dft_xc_eads_df = ce39_dft_xc_eads_df.round(0).astype({dft_xc: 'Int64' for dft_xc in dft_xc_list} | {'Experiment': 'Int64', 'Reaction': str})
final_ce39_eads_df = pd.concat([final_ce39_eads_df, ce39_dft_xc_eads_df[dft_xc_list]], axis=1)

# MAD
mad_dict_final = {m: np.mean(np.abs(final_ce39_eads_df[m] - final_ce39_eads_df['Experiment'])) for m in metrics_methods}
final_ce39_eads_df.loc['MAD'] = ['MAD', 0] + [mad_dict_final[m] for m in metrics_methods]

# weighted MAD
wmad_dict_final = {m: np.mean(np.abs(final_ce39_eads_df[m] - final_ce39_eads_df['Experiment']) * ce39_dataset_df['MAD Weight']) for m in metrics_methods}
final_ce39_eads_df.loc['wMAD'] = ['wMAD', 0] + [wmad_dict_final[m] for m in metrics_methods]

# RMSD
rmsd_dict_final = {m: np.sqrt(np.mean((final_ce39_eads_df[m] - final_ce39_eads_df['Experiment'])**2)) for m in metrics_methods}
final_ce39_eads_df.loc['RMSD'] = ['RMSD', 0] + [rmsd_dict_final[m] for m in metrics_methods]

# weighted RMSD
rmsd_w_dict_final = {m: np.sqrt(np.mean(((final_ce39_eads_df[m] - final_ce39_eads_df['Experiment'])**2) * ce39_dataset_df['MAD Weight'])) for m in metrics_methods}
final_ce39_eads_df.loc['wRMSD'] = ['wRMSD', 0] + [rmsd_w_dict_final[m] for m in metrics_methods]

# Round and cast
astype_map = {'Experiment': 'Int64', 'Reaction': str}
astype_map.update({m: 'Int64' for m in metrics_methods})
final_ce39_eads_df = final_ce39_eads_df.round(0).astype(astype_map)

display(final_ce39_eads_df)

# Combine final_ce39_eads_dict and ce39_dft_xc_eads_dict
final_ce39_eads_dict = {ce39_idx: final_ce39_eads_dict[ce39_idx] | {dft_xc: ce39_dft_xc_eads_dict[ce39_idx][dft_xc] for dft_xc in dft_xc_list} for ce39_idx in ce39_dataset_dict}